# Plots

Thesis figures. Every figure is saved to `plots/latex/` as vector PDF (for `\includegraphics`) and, when a LaTeX installation is available, as PGF (for `\input{}`, fonts follow the document).

In [1]:
import json
import shutil
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
PLOTS_DIR = PROJECT_DIR / "plots" / "latex"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

HAS_LATEX = shutil.which("pdflatex") is not None

mpl.rcParams.update({
    "font.family": "serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "pdf.fonttype": 42,  # embed TrueType so text stays selectable/editable
    "figure.dpi": 110,
})
if HAS_LATEX:
    mpl.rcParams.update({"pgf.texsystem": "pdflatex", "pgf.rcfonts": False})


def save_fig(fig, name):
    """Save as vector PDF, plus PGF if LaTeX is installed."""
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
    saved = [f"{name}.pdf"]
    if HAS_LATEX:
        fig.savefig(PLOTS_DIR / f"{name}.pgf", bbox_inches="tight", backend="pgf")
        saved.append(f"{name}.pgf")
    else:
        print("pdflatex not found: skipped PGF export.")
    print("Saved to", PLOTS_DIR, ":", ", ".join(saved))

In [6]:
with open(DATA_DIR / "market_value_name_cache.json", encoding="utf-8") as f:
    name_cache = pd.DataFrame(json.load(f)).T
name_cache.index.name = "lineup_name"
name_cache["score"] = name_cache["score"].astype(float)

# Must match SIMILARITY_THRESHOLD / MIN_THRESHOLD in 1_player_stats/b_get_market_values.py
SIMILARITY_THRESHOLD = 0.8
MIN_THRESHOLD = 0.5

print(f"{len(name_cache)} cached names, {name_cache['matched_name'].isna().sum()} without a match")
name_cache.head()

# how many unique player names do we have in total?
unique_player_names = name_cache['matched_name'].nunique()
print(f"Unique player names: {unique_player_names}")

8933 cached names, 95 without a match
Unique player names: 8753


In [4]:
# how many unique player names do we have in total?
# use the name_cache df and the lineup_name column
unique_player_names = name_cache['lineup_name'].nunique()

KeyError: 'lineup_name'

## Distribution of fuzzy-matching scores

In [ ]:
scores = name_cache["score"]
bins = np.linspace(0, 1, 21)
counts, edges = np.histogram(scores, bins=bins)
nonzero = counts > 0  # empty bins would put log(0) into the PGF output
Y_MIN = 0.5  # finite bar baseline on the log axis

fig, ax = plt.subplots(figsize=(6.0, 3.6))
ax.bar(edges[:-1][nonzero], counts[nonzero] - Y_MIN, width=np.diff(edges)[nonzero],
       bottom=Y_MIN, align="edge", color="#3b6ea5", edgecolor="white")
ax.axvline(SIMILARITY_THRESHOLD, color="#c0392b", linestyle="--", label=f"Initial threshold ({SIMILARITY_THRESHOLD:.2f})")
ax.axvline(MIN_THRESHOLD, color="#e67e22", linestyle=":", label=f"Minimum threshold ({MIN_THRESHOLD:.2f})")
ax.set_yscale("log")  # exact matches (score 1.0) dwarf all other bins
ax.set_ylim(bottom=Y_MIN)
ax.set_xlabel("Similarity score")
ax.set_ylabel("Number of players (log scale)")
ax.set_xlim(0, 1)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
save_fig(fig, "name_matching_score_histogram")
plt.show()

print(f"Exact matches (1.0): {(scores == 1.0).sum()} | below 1.0: {(scores < 1.0).sum()} | "
      f"below initial threshold: {(scores < SIMILARITY_THRESHOLD).sum()}")